In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
from typing import Union

import pandas as pd
from lippdemo import *
from utils import *

In [2]:
graph = Graph()
print("Adding linkedin data...")
graph = add_linkedin(graph)

print("Adding josh langam data...")
graph = add_josh_langam(graph)

print("Adding extreme talent data...")
graph = add_extreme_talent(graph)

print("Adding crunchbase data...")
graph = add_crunchbase_data(graph)

Adding linkedin data...
Adding josh langam data...
Adding extreme talent data...
Adding /Users/taylormitchell/Code/mew/datasets/demos/data/input/good-signal/intelligentcrazypeople.txt to graph
Adding /Users/taylormitchell/Code/mew/datasets/demos/data/input/good-signal/CPHOF.txt to graph
Adding /Users/taylormitchell/Code/mew/datasets/demos/data/input/good-signal/YC Companies_flattened.txt to graph
Adding /Users/taylormitchell/Code/mew/datasets/demos/data/input/good-signal/International Olympiad Winners.txt to graph
Adding /Users/taylormitchell/Code/mew/datasets/demos/data/input/good-signal/Misc Competition Winners.txt to graph
Adding /Users/taylormitchell/Code/mew/datasets/demos/data/input/good-signal/MLH Top Hackers.txt to graph
Adding /Users/taylormitchell/Code/mew/datasets/demos/data/input/good-signal/Scholarships and Fellowships List.txt to graph
Adding crunchbase data...


In [4]:
# Get counts of people, companies, and investors

all_people_nodes = graph.get_people_nodes()
all_people_node_ids = set(node["id"] for node in all_people_nodes)
all_company_node_ids = set(node["id"] for node in graph.get_companies_nodes())
all_investor_node_ids = set(node["id"] for node in graph.get_investor_nodes())
all_extreme_talent_node_ids = set(node["id"] for node in graph.get_extreme_talent_nodes())
top_investor_node_ids = set(node["id"] for node in graph.get_related_nodes_from_id(vcs_list_id))
print(f"Top investor node ids: {top_investor_node_ids}")

# Get set of top investors each person is connected to
people_to_top_investors_set = {}
for investor_node_id in top_investor_node_ids:
    for relation, node, _ in graph.walk_adjacent(investor_node_id, 1):
        # Add people directly connected to top investors
        if node["id"] in all_people_node_ids:
            people_to_top_investors_set[node["id"]] = people_to_top_investors_set.get(node["id"], set()) | {investor_node_id}
        # Add people connected to companies that have top investors
        if node["id"] in all_company_node_ids:
            for relation, node, _ in graph.walk_adjacent(node["id"], 1):
                if node["id"] in all_people_node_ids:
                    people_to_top_investors_set[node["id"]] = people_to_top_investors_set.get(node["id"], set()) | {investor_node_id}

# Get set of companies each person is connected to
people_to_company_ids = {}
people_to_people_ids = {}
for person_node_id in all_people_node_ids:
    for relation, node, _ in graph.walk_adjacent(person_node_id, 1):
        if node["id"] in all_company_node_ids:
            people_to_company_ids[person_node_id] = people_to_company_ids.get(person_node_id, set()) | {node["id"]}
        if node["id"] in all_people_node_ids:
            people_to_people_ids[person_node_id] = people_to_people_ids.get(person_node_id, set()) | {node["id"]}

important_node_ids = all_people_node_ids | all_company_node_ids | all_investor_node_ids


Top investor node ids: {'0138e526-d92d-49d4-b6b4-39a50016bdfc', '61054593-0aa7-41c2-8bfa-ce40ec8068d6', 'andreessen-horowitz', 'spark-capital', 'bessemer-venture-partners', '7fd4706b-fd74-48e2-bc28-e6088fd16209', 'khosla-ventures', 'ab4c9826-5795-4e6b-97ec-000388659da3', 'accel', '038f8a2f-3864-4627-a547-91953ff08fc6', 'general-catalyst', 'b9a7577c-2e30-41dd-a29e-ae25d727b077', '190718b5-0d5f-445b-a50c-f70bee474b59', 'sequoia-capital', 'kleiner-perkins', '8789dee9-c7db-47c6-9309-1a349e23020a', 'founders-fund', 'bced9934-bf77-4cdb-b5f2-3c57c36e79ee', 'menlo-ventures', 'first-round-capital'}


In [5]:
def sort_key(node):
    return (
        graph.is_extreme_talent(node["id"]) and len(people_to_top_investors_set.get(node["id"], set())) > 0,
        len(people_to_company_ids.get(node["id"], set())) + len(people_to_top_investors_set.get(node["id"], set())),
    )
sorted_people = sorted(all_people_nodes, key=lambda n: sort_key(n), reverse=True)

In [6]:
# Unless a relation is between important nodes (people, companies, or investors)
# Flag it as an ideapad attribute
relations = list(graph._graph["relationsById"].values())
relations_with_ideapad_attr = set(r["id"] for r in graph.get_relations_with_to_id(ideapad_show_as_attribute_node_id))
relations_with_ideapad_none = set(r["id"] for r in graph.get_relations_with_to_id(ideapad_show_as_none_node_id))

for relation in relations:
    if relation["id"] in relations_with_ideapad_attr:
        continue
    if relation["id"] in relations_with_ideapad_none:
        continue
    from_node = graph.get_node(relation["fromId"])
    to_node = graph.get_node(relation["toId"])
    if not from_node or not to_node:
        continue
    from_is_important_node = from_node["id"] in important_node_ids
    to_is_important_node = to_node["id"] in important_node_ids
    # If it's e.g. a relation b/w a person and company, leave as-is
    if from_is_important_node and to_is_important_node:
        continue
    # If it's a relation about a person/company, make it an ideapad attribute
    if from_is_important_node:
        make_ideapad_attribute(graph, relation)
# Hide all nodes that aren't important
nodes = list(graph._graph["nodesById"].values())
for node in nodes:
    if node["id"] not in important_node_ids:
        make_ideapad_none(graph, node)

In [8]:
# Create lite version
graph_lite = filter_by_people(graph, set(node["id"] for node in sorted_people[:300]))
# graph_lite = graph

assign_canonical_relation(graph_lite)
assert_graph_integrity(graph_lite._graph)
lite_path = os.path.join(output_dir, f"lippdemo-lite.json")
with open(lite_path, "w") as f:
    json.dump(graph_lite.to_dict(), f)

# print(f"Saved full version to {full_path}")
print(f"Saved lite version to {lite_path}")

Saved lite version to /Users/taylormitchell/Code/mew/datasets/demos/data/output/lippdemo-lite.json
